In [ ]:
# pip install gymnasium IPython torch

In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

In [3]:
# Create environment and Q-table

# training w/o rendering
env = gym.make("FrozenLake-v1", is_slippery=False)

number_of_states = env.observation_space.n
number_of_actions = env.action_space.n

q_table = np.zeros((number_of_states, number_of_actions), dtype=float)
rng = np.random.default_rng(42)
env.action_space.seed(42)

# save the map to draw later
lake_map = env.unwrapped.desc.astype(str)
number_of_rows, number_of_columns = lake_map.shape


# each row to represent a state, each column to represent an action

# hyperparameters ( RL --> learning parameters )
learning_rate = 0.8
discount_factor = 0.95
epsilon = 1 #exploration vs exploitation
minimum_epsilon = 0.05
epsilon_decay = 0.99
episodes = 5000


# collect data for viz

episode_rewards = []
episode_successes = []
epsilon_history = []
state_visits = np.zeros(number_of_states, dtype = int)


# store one Q-learning example
first_update_example = None
bootstrapped_update_example = None

# actions = left, down, right, up
action_arrows = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}


# train the agent

for episode in range(episodes):
    state, info = env.reset(seed = 42 if episode == 0 else None)
    
    terminated = False
    truncated = False
    episode_return = 0.0
    
    while not (terminated or truncated):
        state_visits[state] += 1
        
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(q_table[state]))
        
        next_state, reward, terminated, truncated, info = env.step(action)
        
        old_q_value = q_table[state, action]
        
        future_value = np.max(q_table[next_state])
        
        if terminated:
            discounted_future_value = 0.0
        else:
            discounted_future_value = discount_factor * future_value
        
        target = reward + discounted_future_value
        
        new_q_value = old_q_value + learning_rate * (target - old_q_value)
        q_table[state, action] = new_q_value
        
        if first_update_example is None:
            first_update_example = {
                "state": state,
                "action": action,
                "next_state": next_state,
                "old_q": old_q_value,
                "reward": reward,
                "discounted_future": discounted_future_value,
                "target": target,
                "new_q": new_q_value
            }
        
        if bootstrapped_update_example is None and not terminated and future_value > 0:
            bootstrapped_update_example = {
                "state": state,
                "action": action,
                "next_state": next_state,
                "old_q": old_q_value,
                "reward": reward,
                "discounted_future": discounted_future_value,
                "target": target,
                "new_q": new_q_value
            }
        
        state = next_state
        episode_return += reward
        
    episode_rewards.append(episode_return)
    episode_successes.append(int (state == number_of_states - 1))
    epsilon_history.append(epsilon)
    
    epsilon = max(minimum_epsilon, epsilon * epsilon_decay)
        
env.close()
        
        